# ✈️ TRIP.com Price Crawler

Chạy lần lượt các cell từ trên xuống: **① Cấu hình → ② Đọc input → ③ Crawl → ④ Xem kết quả**.

> Trip chạy **BROWSER-ONLY** (room API ký từng request — không replay được như Agoda). Stack anti-detect giống Agoda: **Camoufox** (Firefox + humanize + geoip), bỏ skeleton rỗng lazy-load, scroll kích hoạt `getHotelRoomList`, fallback DOM trước khi soft-block.

**Đổi nguồn input:** sửa `INPUT_MODE` ở cell ① — `"gsheet"` (Google Sheet online) hoặc `"offline"` (file CSV/XLSX trên máy).

**Format file offline:** chỉ cần **3 cột đầu theo đúng thứ tự** `hotel_name, hotel_url, room_type` (tên cột không quan trọng, chỉ cần đúng thứ tự). Có file mẫu ở `input/TEMPLATE_hotels.csv`.

**Output** nằm trong `results/trip/`:
- `FINAL_<YYYYMMDD>.csv` — kết quả cuối
- `TEMP_trip.csv` — checkpoint: lỡ tắt giữa chừng, chạy lại cell ③ sẽ tự resume phần chưa xong — hotel **chưa từng cào** sẽ chạy trước, hotel còn NA/SOLD OUT retry sau

In [1]:
# ════════════════ ① CẤU HÌNH ════════════════

# ── Nguồn input: "gsheet" (online) hoặc "offline" (file trên máy) ──
INPUT_MODE = "gsheet"

# Dùng khi INPUT_MODE = "gsheet" (gid của tab được tự lấy từ URL)
GSHEET_URL = "https://docs.google.com/spreadsheets/d/1g_S06QeEAWnCTHYXGH0Nn4Mcb3FCT_-uIS1jUm4GGkw/edit?gid=607908359#gid=607908359"

# Dùng khi INPUT_MODE = "offline" — đường dẫn tuyệt đối, hoặc tương đối so với 31.crawl-tool
OFFLINE_FILE = "input/trip_hotels.csv"

# ── Tham số crawl ──
WEEKS      = 6      # số tuần cần crawl
MAX_HOTELS = 0      # 0 = crawl tất cả; đặt 5 để test nhanh 5 khách sạn đầu
SHARD      = ""     # "" = không chia; "1/2" = chạy phần 1 trong 2 phần (chạy lần lượt 1/2 rồi 2/2)

In [2]:
# ════════════════ ② ĐỌC INPUT ════════════════
import os, sys

if "ROOT" not in globals():                    # giữ nguyên ROOT khi chạy lại cell
    ROOT = os.path.abspath("")                 # .../31.crawl-tool (nơi đặt notebook này)
assert os.path.isdir(os.path.join(ROOT, "crawler")), (
    f"Không tìm thấy package `crawler` trong {ROOT} — hãy mở notebook từ thư mục 31.crawl-tool")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import crawler
from crawler.hotels_io import read_hotels

if INPUT_MODE == "gsheet":
    INPUT = GSHEET_URL
    print("📡 Input: Google Sheet online")
else:
    INPUT = OFFLINE_FILE if os.path.isabs(OFFLINE_FILE) else os.path.join(ROOT, OFFLINE_FILE)
    assert os.path.exists(INPUT), f"Không tìm thấy file: {INPUT}"
    print(f"📁 Input: file offline — {INPUT}")

hotels = read_hotels(INPUT)
print(f"✅ Đọc được {len(hotels)} khách sạn. 5 dòng đầu:")
for name, url, room in hotels[:5]:
    print(f"   • {name} — {room}")

📡 Input: Google Sheet online
✅ Đọc được 55 khách sạn. 5 dòng đầu:
   • Halais Hotel — Superior Room
   • Minasi HanoiOi Hotel — Premier Double Bed Room
   • Muong Thanh Hanoi Centre Hotel — Superior King Room
   • Mercure Hanoi La Gare Hotel — Classic Double Bed Room With City View
   • REY Hotel Hanoi — Standard Twin Room


In [3]:
# (TÙY CHỌN) Tải Google Sheet về file offline — lần sau chỉ cần đổi INPUT_MODE = "offline"
import pandas as pd
from crawler.hotels_io import _gsheet_url

os.makedirs(os.path.join(ROOT, "input"), exist_ok=True)
dest = os.path.join(ROOT, "input", "trip_hotels.csv")
pd.read_csv(_gsheet_url(GSHEET_URL)).to_csv(dest, index=False, encoding="utf-8-sig")
print(f"💾 Đã lưu bản offline: {dest}")

💾 Đã lưu bản offline: /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/input/trip_hotels.csv


In [4]:
# ════════════════ ③ CRAWL ════════════════
import subprocess
from crawler import envcheck

# Camoufox BẮT BUỘC cho Trip (v0.5+): anti-detect Firefox + humanize + geoip — giống stack Agoda.
# Playwright PHẢI là 1.59.x — 1.60 crash Camoufox, 1.61+ lỗi isMobile/setDefaultViewport.
if not envcheck.has("camoufox"):
    print("📦 camoufox chưa có — đang pip install…", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "camoufox[geoip]"])
pw_ok, pw_ver = envcheck.playwright_camoufox_ok()
if not pw_ok:
    print(f"📦 playwright {pw_ver} không tương thích Camoufox — pin 1.59.x…", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "playwright>=1.59,<1.60"])
    subprocess.check_call([sys.executable, "-m", "playwright", "install", "chromium"])
if not envcheck.camoufox_browser_ok():
    print("📦 camoufox browser binary missing — đang fetch…", flush=True)
    subprocess.check_call([sys.executable, "-m", "camoufox", "fetch"])
    assert envcheck.camoufox_browser_ok(), "camoufox fetch xong nhưng binary vẫn chưa thấy"

OUTDIR = os.path.join(ROOT, "results", "trip")
os.makedirs(OUTDIR, exist_ok=True)
os.chdir(OUTDIR)                     # output (FINAL_*.csv, TEMP_trip.csv) nằm ở đây

kwargs = dict(
    site="trip",                     # browser-only qua Camoufox (Trip không replay trực tiếp được)
    input=INPUT,
    weeks=WEEKS,
)
if MAX_HOTELS:
    kwargs["max"] = MAX_HOTELS
if SHARD:
    kwargs["shard"] = SHARD

await crawler.arun(**kwargs)         # notebook cho phép await trực tiếp

📂 Resume: 55 rows from TEMP_trip.csv
🚀 TRIP crawl | 55 hotels × 6w | browser-only | engine=camoufox | W1=2026-07-28
🦊 Camoufox ready (humanize=True geoip=True) — browser navs use anti-detect Firefox
✔️  1/55 Halais Hotel — complete, skip
✔️  2/55 Minasi HanoiOi Hotel — complete, skip
✔️  3/55 Muong Thanh Hanoi Centre Hotel — complete, skip
✔️  4/55 Mercure Hanoi La Gare Hotel — complete, skip
✔️  5/55 REY Hotel Hanoi — complete, skip
✔️  6/55 La Passion Premium Cau Go — complete, skip
✔️  7/55 Bespoke Trendy Hotel Hanoi (Formerly Hanoi La Siesta Hotel Trendy) — complete, skip
✔️  8/55 Nesta Hotel Hanoi — complete, skip
✔️  9/55 Silk Path Boutique Hanoi — complete, skip
✔️  10/55 La Siesta Premium Hang Be — complete, skip
✔️  11/55 Hanoi le Jardin Hotel & Spa — complete, skip
✔️  12/55 Hotel du Lac Hanoi — complete, skip
✔️  13/55 L'Signature Hotel & Spa — complete, skip
✔️  14/55 Aira Boutique Hanoi Hotel & Spa — complete, skip
✔️  15/55 Melia Hanoi — complete, skip
✔️  16/55 Movenpick

'FINAL_20260723.csv'

In [5]:
# ════════════════ ④ XEM KẾT QUẢ ════════════════
import glob
import pandas as pd

OUTDIR = os.path.join(ROOT, "results", "trip")
files = sorted(glob.glob(os.path.join(OUTDIR, "FINAL_*.csv")))
assert files, "Chưa có file FINAL nào — hãy chạy cell ③ trước."
latest = files[-1]
df = pd.read_csv(latest)
print(f"📄 {latest} — {len(df)} dòng")
df.head(20)

📄 /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/results/trip/FINAL_20260723.csv — 55 dòng


,hotel_name,room_type,price_w1,price_w2,price_w3,price_w4,price_w5,price_w6
0,Halais Hotel,Superior Room,"VND 1,328,594","VND 1,082,582","VND 1,082,582","VND 1,082,582","VND 1,082,582","VND 1,082,582"
1,Minasi HanoiOi Hotel,Premier Double Bed Room,"VND 2,173,913","VND 1,467,391","VND 1,467,391","VND 1,467,391","VND 1,467,391","VND 1,467,391"
2,Muong Thanh Hanoi Centre Hotel,Superior King Room,"VND 1,400,000","VND 1,282,544","VND 1,282,544","VND 1,282,544","VND 1,282,544","VND 2,500,000"
3,Mercure Hanoi La Gare Hotel,Classic Double Bed Room With City View,"VND 1,723,680","VND 1,723,680","VND 1,723,680","VND 1,723,680","VND 1,723,680","VND 1,723,680"
4,REY Hotel Hanoi,Standard Twin Room,"VND 2,700,000","VND 2,300,000","VND 2,500,000","VND 2,300,000","VND 1,725,000","VND 1,725,000"
5,La Passion Premium Cau Go,Grand Double Room,"VND 3,660,163","VND 8,543,630","VND 8,774,403","VND 8,774,403","VND 8,774,403","VND 8,775,000"
6,Bespoke Trendy Hotel Hanoi (Formerly Hanoi La ...,Cozy Deluxe Room,"VND 4,000,000","VND 4,500,000","VND 1,585,698","VND 1,596,153","VND 1,596,153","VND 1,732,967"
7,Nesta Hotel Hanoi,Superior Twin Room Non Smoking,"VND 1,311,428","VND 1,213,912","VND 1,213,912","VND 1,277,802","VND 1,244,175","VND 1,277,802"
8,Silk Path Boutique Hanoi,Superior Room,"VND 1,389,803","VND 1,389,803","VND 1,389,803","VND 1,389,803","VND 3,318,400","VND 1,389,803"
9,La Siesta Premium Hang Be,Superior Room No Window,"VND 2,668,050","VND 3,430,350","VND 3,049,200","VND 3,201,660","VND 2,286,900","VND 2,286,900"
